# Ternary Interpolation Demo (Al-Si-Mg)

This notebook demonstrates the binary-to-ternary interpolation workflow for the Al-Si-Mg system. For use on the entire gliquid binary dataset, check out the [the online version](https://viz.whsunresearch.group/gliquid/ternary-interpolation/)

1. Define a helper for ternary interpolation/plotting.
2. Intialize binary fitted parameters.
3. Generate an interactive ternary plot.

In [2]:
## UNCOMMENT THIS BLOCK FOR GOOGLE COLAB
# !git clone https://github.com/willwerj/gliquid_python.git
# %cd /content/gliquid_python
# !pip install -q .
# from pathlib import Path
# import gliquid.config as cfg
# cfg.set_data_dir(Path("/content/gliquid_python/data").resolve())

In [ ]:
import os
from pathlib import Path
import gliquid.config as cfg

# Resolve the repository root even when this notebook is launched from notebooks/.
_project_root = Path.cwd().resolve()
for _candidate in [_project_root, *_project_root.parents]:
    if (_candidate / 'pyproject.toml').exists() and (_candidate / 'src' / 'gliquid').exists():
        _project_root = _candidate
        break

cfg.set_project_root(_project_root)
cfg.set_data_dir(_project_root / 'data')

print(f"Project root : {cfg.project_root}")
print(f"Data directory: {cfg.data_dir}")

# Materials Project API key — required for fetching DFT convex hull data.
# Replace with your own key from https://materialsproject.org/api
os.environ["NEW_MP_API_KEY"] = "YOUR_API_KEY_HERE"

Project root : C:\Users\AbrarRauf\Desktop\gliquid_python_pr_clean
Data directory: C:\Users\AbrarRauf\Desktop\gliquid_python_pr_clean\data


## 1 - Define the interpolation helper

This function formats fitted binary parameters to match the sorted ternary axis order used by `hsx_ternary.py`, interpolates the ternary free-energy surface, and optionally writes an HTML output. It also returns the initialized plotter so the binary TX figures generated during ternary processing can be inspected.


In [4]:
import plotly.offline as ploff
from pathlib import Path
from gliquid.hsx_ternary import ternary_gtx_plotter, ordered_binary_systems, invert_substrings


def _ordered_binary_params(tern_sys, binary_data):
    """Return L parameters keyed in hsx_ternary's cyclic binary order."""
    target_systems = ordered_binary_systems(tern_sys)
    L_dict = {}
    fit_or_pred = {}

    for target_sys in target_systems:
        source_sys = target_sys
        invert = False
        if source_sys not in binary_data:
            source_sys = invert_substrings(target_sys)
            invert = True
        if source_sys not in binary_data:
            raise ValueError(f"Missing parameters for required binary {target_sys}.")

        data = binary_data[source_sys]
        params = list(map(float, data['params']))
        if invert:
            params[2:] = [-p for p in params[2:]]

        L_dict[target_sys] = params
        fit_or_pred[target_sys] = data.get('fit_or_pred', data.get('source', 'fit'))

    return L_dict, fit_or_pred


def plot_ternary_system(
    tern_sys,
    binary_data,
    param_format='comb-exp',
    interp_type='linear',
    temp_slider=(0, 300),
    t_incr=5,
    delta=0.025,
    save_html=False,
):
    """Interpolate ternary liquid free energy from binary parameters."""
    if isinstance(tern_sys, str):
        tern_sys = tern_sys.split('-')
    tern_sys = sorted(tern_sys)

    binary_L_dict, fit_or_pred = _ordered_binary_params(tern_sys, binary_data)

    plotter = ternary_gtx_plotter(
        tern_sys,
        interp_type=interp_type,
        param_format=param_format,
        L_dict=binary_L_dict,
        fit_or_pred=fit_or_pred,
        temp_slider=list(temp_slider),
        T_incr=t_incr,
        delta=delta,
    )
    plotter.interpolate()
    plotter.process_data()
    tern_fig = plotter.plot_ternary()

    if save_html:
        out_file = Path(cfg.project_root) / 'figures' / f"{''.join(tern_sys)}_system.html"
        out_file.parent.mkdir(parents=True, exist_ok=True)
        ploff.plot(tern_fig, filename=str(out_file), auto_open=False)
        print(f"Saved ternary figure: {out_file}")

    return tern_fig, plotter


c:\Users\AbrarRauf\miniconda3\envs\gliq-clean-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



## 2 - Define binary fitted parameters

Here are parameters for three of the binary systems in our database: Al-Si, Al-Mg and Mg-Si. All of these are fitted using the 'comb-exp' model, which is presently the default model we use for ternary interpolations

In [5]:
fitted_data = {
    'Al-Si': {'params': [-13430, -1.516, -966.7, 0]},
    'Al-Mg': {'params': [804.7, 0.08031, 258.5, 0]},
    'Mg-Si': {'params': [-45190, 9.047, 1482, 0]},
}

for key, value in fitted_data.items():
    print(f"{key}: {value}")

Al-Si: {'params': [-13430, -1.516, -966.7, 0]}
Al-Mg: {'params': [804.7, 0.08031, 258.5, 0]}
Mg-Si: {'params': [-45190, 9.047, 1482, 0]}


## 3 - Interpolate and plot the ternary system

Run the interpolation and display the interactive ternary diagram. The helper returns both the ternary figure and the initialized plotter. Set `save_html=True` to also write an HTML file in the `figures` directory.


In [6]:
ternary_figure, ternary_plotter = plot_ternary_system(
    'Al-Si-Mg',
    fitted_data,
    param_format='comb-exp',
    interp_type='linear',
    t_incr=5, # Sets the temperature resolution - smaller values increase resolution but also increase computation time
    delta=0.01, # Sets the composition resolution - smaller values increase resolution but also increase computation time
    save_html=False,
)
ternary_figure.show()


Composition map: x0: Mg, x1: Si


Retrieving ThermoDoc documents: 100%|██████████| 780/780 [00:00<?, ?it/s]


Caching ternary DFT entry data as C:\Users\AbrarRauf\Desktop\gliquid_python_pr_clean\data\Al-Mg-Si_ENTRIES_MP_GGA.json...


Retrieving ThermoDoc documents: 100%|██████████| 50/50 [00:00<?, ?it/s]


Al: H_liq = 10710.0 J/mol, S_liq = 11.4730 J/(mol·K), T_fusion = 933.5 K, polymorphs = 1
Mg: H_liq = 8480.0 J/mol, S_liq = 9.1874 J/(mol·K), T_fusion = 923.0 K, polymorphs = 1

No cached binary phase data found!
MPDS_API_KEY not found in environment variables. Proceeding without binary phase data.


...

Mg: H_liq = 8480.0 J/mol, S_liq = 9.1874 J/(mol·K), T_fusion = 923.0 K, polymorphs = 1
Si: H_liq = 50210.0 J/mol, S_liq = 29.7629 J/(mol·K), T_fusion = 1687.0 K, polymorphs = 1

No cached binary phase data found!
MPDS_API_KEY not found in environment variables. Proceeding without binary phase data.


Retrieving ThermoDoc documents: 100%|██████████| 55/55 [00:00<?, ?it/s]


Al: H_liq = 10710.0 J/mol, S_liq = 11.4730 J/(mol·K), T_fusion = 933.5 K, polymorphs = 1
Si: H_liq = 50210.0 J/mol, S_liq = 29.7629 J/(mol·K), T_fusion = 1687.0 K, polymorphs = 1

No cached binary phase data found!
MPDS_API_KEY not found in environment variables. Proceeding without binary phase data.
Initialization complete


Evaluating lower hull over temperature intervals: 100%|██████████| 439/439 [01:46<00:00,  4.11it/s] 


Lower hull evaluation and post processing time:: 106.95320749282837 seconds for temperature increment of 5


## 4 - Inspect the binary phase diagrams used for interpolation

`ternary_gtx_plotter.process_data()` also builds binary TX figures used to interpolate the ternary figure. These figures are available from `ternary_plotter.bin_fig_list`.


In [7]:
if ternary_plotter.bin_fig_list:
    for binary_figure in ternary_plotter.bin_fig_list:
        binary_figure.show()
else:
    print('No binary figures were generated. Check fit_or_pred and binary cache availability.')
